# Verification Platform 260715版 — Gradio UI

## ユーザーマニュアル

- [計算フロー（タイプ1・2）](https://iguchi-lab.github.io/pyhees-jjj/%E8%A8%88%E7%AE%97%E3%83%95%E3%83%AD%E3%83%BC_%E3%82%BF%E3%82%A4%E3%83%971%E3%83%BB2.html)
- [最低風量・最低電力の直接入力](https://iguchi-lab.github.io/pyhees-jjj/%E6%9C%80%E4%BD%8E%E9%A2%A8%E9%87%8F_%E6%9C%80%E4%BD%8E%E9%9B%BB%E5%8A%9B_%E7%9B%B4%E6%8E%A5%E5%85%A5%E5%8A%9B.html)

### 入力方法

- **(1) UI入力**: 第1階層で「基本設定・暖房・冷房・換気」、第2階層で入力セクションを選びます。
- **(2) ローカル**: 端末内のJSONファイルをアップロードします。複数ファイルを指定できます。
- 複数の計算を実行した場合、入力内容とグラフには最後に計算した結果だけを表示します。

<a href="https://colab.research.google.com/github/iguchi-lab/Verification-Platform/blob/feat/gradio-ui-260715/Verification_Platform_260715_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 注意

- `share=True` のURLは一時的で、URLを知る人がアクセスできます。
- 機密情報を入力しないでください。ランタイム終了時にUIも停止します。


In [ ]:
#@title セットアップ（260715 Gradio版）

# pyhees-jjj固定コミット: 0f91ba8381df1b4960557b92b39339385cc9009f
!pip3 install git+https://github.com/iguchi-lab/pyhees-jjj.git@0f91ba8381df1b4960557b92b39339385cc9009f
!pip3 install -q japanize-matplotlib
!pip3 install -q "gradio>=6,<7"


In [ ]:
#@title Gradio UIを起動（260715版）
import ast
import base64
import contextlib
import io
import html
import json
import os
import re
import sys
import traceback
import time
import tempfile
import zipfile
from pathlib import Path
from collections import OrderedDict, defaultdict

import gradio as gr
import jjjexperiment.main
import matplotlib.pyplot as plt
from jjjexperiment.constants import version_info

_RELEASE_VERSION = "260715"
_FORM_SOURCE = base64.b64decode("...").decode("utf-8")
_GRAPH_SOURCE = base64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IGphcGFuaXplX21hdHBsb3RsaWIKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApmcm9tIGpqamV4cGVyaW1lbnQuY29uc3RhbnRzIGltcG9ydCB2ZXJzaW9uX2luZm8KCnByZWZpeF9maWxlbmFtZSA9IGlucHV0X2RhdGFbJ2Nhc2VfbmFtZSddICsgdmVyc2lvbl9pbmZvKCkKCmRmX291dHB1dDIgICA9IHBkLnJlYWRfY3N2KHByZWZpeF9maWxlbmFtZSArICdfb3V0cHV0Mi5jc3YnLCAgIGVuY29kaW5nID0gJ2NwOTMyJywgcGFyc2VfZGF0ZXMgPSBUcnVlLCBpbmRleF9jb2wgPSAwKQpkZl9vdXRwdXQ1X0MgPSBwZC5yZWFkX2NzdihwcmVmaXhfZmlsZW5hbWUgKyAnX0Nfb3V0cHV0NS5jc3YnLCBlbmNvZGluZyA9ICdjcDkzMicsIHBhcnNlX2RhdGVzID0gVHJ1ZSwgaW5kZXhfY29sID0gMCkKZGZfb3V0cHV0NV9IID0gcGQucmVhZF9jc3YocHJlZml4X2ZpbGVuYW1lICsgJ19IX291dHB1dDUuY3N2JywgZW5jb2RpbmcgPSAnY3A5MzInLCBwYXJzZV9kYXRlcyA9IFRydWUsIGluZGV4X2NvbCA9IDApCgp3aW50ZXJfZGZfb3V0cHV0MiAgID0gICBkZl9vdXRwdXQyLmxvY1snMjAyMy0wMi0wNSAwMDowMDowMCcgOiAnMjAyMy0wMi0xMyAwMDowMDowMCddCndpbnRlcl9kZl9vdXRwdXQ1X0ggPSBkZl9vdXRwdXQ1X0gubG9jWycyMDIzLTAyLTA1IDAwOjAwOjAwJyA6ICcyMDIzLTAyLTEzIDAwOjAwOjAwJ10Kc3VtbWVyX2RmX291dHB1dDIgICA9ICAgZGZfb3V0cHV0Mi5sb2NbJzIwMjMtMDgtMDYgMDA6MDA6MDAnIDogJzIwMjMtMDgtMTQgMDA6MDA6MDAnXQpzdW1tZXJfZGZfb3V0cHV0NV9DID0gZGZfb3V0cHV0NV9DLmxvY1snMjAyMy0wOC0wNiAwMDowMDowMCcgOiAnMjAyMy0wOC0xNCAwMDowMDowMCddCgpzb3J0ZWRfSF9kZl9vdXRwdXQyID0gZGZfb3V0cHV0Mi5zb3J0X3ZhbHVlcygncV9oc19IX2RfdCBbV2gvaF0nLCBhc2NlbmRpbmcgPSBGYWxzZSkKc29ydGVkX0hfZGZfb3V0cHV0Mi5sb2Nbc29ydGVkX0hfZGZfb3V0cHV0MlsncV9oc19IX2RfdCBbV2gvaF0nXSA8PSAwLjBdID0gbnAubmFuCgpkZl9vdXRwdXQyWydxX2hzX0NfZF90IFtXaC9oXSddID0gZGZfb3V0cHV0MlsncV9oc19DU19kX3QgW1doL2hdJ10gKyBkZl9vdXRwdXQyWydxX2hzX0NMX2RfdCBbV2gvaF0nXQpzb3J0ZWRfQ19kZl9vdXRwdXQyID0gZGZfb3V0cHV0Mi5zb3J0X3ZhbHVlcygncV9oc19DX2RfdCBbV2gvaF0nLCBhc2NlbmRpbmcgPSBGYWxzZSkKc29ydGVkX0NfZGZfb3V0cHV0MlsncV9oc19DX2RfdCBbV2gvaF0nXSA9IHNvcnRlZF9DX2RmX291dHB1dDJbJ3FfaHNfQ1NfZF90IFtXaC9oXSddICsgc29ydGVkX0NfZGZfb3V0cHV0MlsncV9oc19DTF9kX3QgW1doL2hdJ10Kc29ydGVkX0NfZGZfb3V0cHV0Mi5sb2Nbc29ydGVkX0NfZGZfb3V0cHV0MlsncV9oc19DX2RfdCBbV2gvaF0nXSA8PSAwLjBdID0gbnAubmFuCgojIyMjSGVhdGluZyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMKCnBsdC5maWd1cmUoZmlnc2l6ZT0oMjUsIDcpKQoKcGx0LnN1YnBsb3QoMiwgMiwgMSkKcGx0LnBsb3Qod2ludGVyX2RmX291dHB1dDIuaW5kZXgsCiAgICAgICAgIHdpbnRlcl9kZl9vdXRwdXQyWyJUaGV0YV9leF9kX3QgW+KEg10iXSwKICAgICAgICAgbGFiZWwgPSAi5aSW5rCX5ripW+KEg10iLCBjb2xvciA9ICdsaWdodGJsdWUnKQpwbHQucGxvdCh3aW50ZXJfZGZfb3V0cHV0NV9ILmluZGV4LAogICAgICAgICB3aW50ZXJfZGZfb3V0cHV0NV9IWyJYX2V4X2RfdCJdICogMTAwMCwKICAgICAgICAgbGFiZWwgPSAi5aSW5rCX57W25a++5rm/5bqmW2cva2cnXSIsIGNvbG9yID0gJ29yYW5nZScpCnBsdC55bGFiZWwoIua4qeW6plvihINd44CB57W25a++5rm/5bqmW2cva2cnXSIpCnBsdC55bGltKC0xMCwgNDApCnBsdC5ncmlkKCkKcGx0LmxlZ2VuZCgpCgpwbHQuc3VicGxvdCgyLCAyLCAyKQpwbHQucGxvdCh3aW50ZXJfZGZfb3V0cHV0Mi5pbmRleCwKICAgICAgICAgd2ludGVyX2RmX291dHB1dDJbInFfaHNfSF9kX3QgW1doL2hdIl0gLyAxMDAwLAogICAgICAgICBsYWJlbCA9ICLlh6bnkIbnhrHph49ba1doL2hdIiwgY29sb3IgPSAncmVkJykKcGx0LnBsb3Qod2ludGVyX2RmX291dHB1dDIuaW5kZXgsCiAgICAgICAgIHdpbnRlcl9kZl9vdXRwdXQyWyJFX0VfSF9kX3QgW2tXaC9oXSJdLAogICAgICAgICBsYWJlbCA9ICJBQytGQU7mtojosrvpm7vliptba1doL2hdIiwgY29sb3IgPSAnYmx1ZScpCnBsdC5wbG90KHdpbnRlcl9kZl9vdXRwdXQyLmluZGV4LAogICAgICAgICB3aW50ZXJfZGZfb3V0cHV0MlsiRV9FX2Zhbl9IX2RfdCBba1doL2hdIl0sCiAgICAgICAgIGxhYmVsID0gIkZBTua2iOiyu+mbu+WKm1trV2gvaF0iLCBjb2xvciA9ICdncmVlbicpCnBsdC55bGFiZWwoIuWHpueQhueGsemHj1trV2gvaF3jgIHmtojosrvpm7vliptba1doL2hdIikKcGx0LnlsaW0oMCwgMTApCnBsdC5ncmlkKCkKcGx0LmxlZ2VuZCgpCgpwbHQuc3VicGxvdCgyLCAyLCAzKQpwbHQucGxvdCh3aW50ZXJfZGZfb3V0cHV0Mi5pbmRleCwKICAgICAgICAgd2ludGVyX2RmX291dHB1dDJbIkVfVVRfSF9kX3QgW01KL2hdIl0sCiAgICAgICAgIGxhYmVsID0gIuWHpueQhueGsemHj1trV2gvaF0iLCBjb2xvciA9ICdyZWQnKQpwbHQueWxhYmVsKCLmnKrlh6bnkIbosqDojbfvvIjkuIDmrKHjgqjjg43jg6vjgq7jg7znm7jlvZPliIbvvIlbTUovaF0iKQpwbHQueWxpbSgwLCAxMCkKcGx0LmdyaWQoKQpwbHQubGVnZW5kKCkKCnBsdC5zdWJwbG90KDIsIDIsIDQpCnBsdC5wbG90KHdpbnRlcl9kZl9vdXRwdXQyLmluZGV4LAogICAgICAgICB3aW50ZXJfZGZfb3V0cHV0MlsicV9oc19IX2RfdCBbV2gvaF0iXSAvIDEwMDAgLyB3aW50ZXJfZGZfb3V0cHV0MlsiRV9FX0hfZF90IFtrV2gvaF0iXSwKICAgICAgICAgbGFiZWwgPSAic0NPUFstXSIsIGNvbG9yID0gJ2JsYWNrJykKcGx0LnlsYWJlbCgic0NPUFstXSIpCnBsdC55bGltKDAsIDEwKQpwbHQuZ3JpZCgpCnBsdC5sZWdlbmQoKQpwbHQuc2hvdygpCgojIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMKCnBsdC5maWd1cmUoZmlnc2l6ZT0oMjUsIDcpKQoKcGx0LnN1YnBsb3QoMiwgNCwgMSkKcGx0LnBsb3QocmFuZ2UoODc2MCksIHNvcnRlZF9IX2RmX291dHB1dDJbJ3FfaHNfSF9kX3QgW1doL2hdJ10gLyAxMDAwLCBjb2xvciA9ICdvcmFuZ2UnKQpwbHQueWxhYmVsKCLlh6bnkIbnhrHph49ba1doL2hdIikKcGx0LnhsYWJlbCgi5pmC6ZaTW2hdIikKcGx0LnlsaW0oMCwgMTApCnBsdC5ncmlkKCkKCnBsdC5zdWJwbG90KDIsIDQsIDIpCnBsdC5wbG90KHJhbmdlKDg3NjApLCBzb3J0ZWRfSF9kZl9vdXRwdXQyWydUaGV0YV9oc19IX291dF9kX3QgW+KEg10nXSwgY29sb3IgPSAnb3JhbmdlJykKcGx0LnlsYWJlbCgi5a6k5YaF5qmf5Ye65Y+j5rip5bqmW+KEg10iKQpwbHQueGxhYmVsKCLmmYLplpNbaF0iKQpwbHQueWxpbSgwLCA1MCkKcGx0LmdyaWQoKQoKcGx0LnN1YnBsb3QoMiwgNCwgMykKcGx0LnBsb3QocmFuZ2UoODc2MCksIHNvcnRlZF9IX2RmX291dHB1dDJbJ1RoZXRhX2hzX0hfaW5fZF90IFvihINdJ10sIGNvbG9yID0gJ29yYW5nZScpCnBsdC55bGFiZWwoIuWupOWGheapn+WFpeWPo+a4qeW6plvihINdIikKcGx0LnhsYWJlbCgi5pmC6ZaTW2hdIikKcGx0LnlsaW0oMCwgNTApCnBsdC5ncmlkKCkKCnBsdC5zdWJwbG90KDIsIDQsIDQpCnBsdC5wbG90KHJhbmdlKDg3NjApLCBzb3J0ZWRfSF9kZl9vdXRwdXQyWydWX2hzX3N1cHBseV9IX2RfdCBbbTMvaF0nXSwgY29sb3IgPSAnb3JhbmdlJykKcGx0LnlsYWJlbCgi5L6b57Wm6aKo6YePW20zL2hdIikKcGx0LnhsYWJlbCgi5pmC6ZaTW2hdIikKcGx0LnlsaW0oMCwgMjAwMCkKcGx0LmdyaWQoKQoKcGx0LnN1YnBsb3QoMiwgNCwgNSkKcGx0LnBsb3QocmFuZ2UoODc2MCksIHNvcnRlZF9IX2RmX291dHB1dDJbJ0VfSF9kX3QgW01KL2hdJ10sIGNvbG9yID0gJ29yYW5nZScpCnBsdC55bGFiZWwoIuS4gOasoeOCqOODjeODq+OCruODvOa2iOiyu+mHj1tNSi9oXSIpCnBsdC54bGFiZWwoIuaZgumWk1toXSIpCnBsdC55bGltKDAsIDUwKQpwbHQuZ3JpZCgpCgpwbHQuc3VicGxvdCgyLCA0LCA2KQpwbHQucGxvdChyYW5nZSg4NzYwKSwgc29ydGVkX0hfZGZfb3V0cHV0MlsnRV9VVF9IX2RfdCBbTUovaF0nXSwgY29sb3IgPSAnb3JhbmdlJykKcGx0LnlsYWJlbCgi5pyq5Yem55CG6LKg6I2377yI5LiA5qyh44Ko44ON44Or44Ku44O855u45b2T5YiG77yJW01KL2hdIikKcGx0LnhsYWJlbCgi5pmC6ZaTW2hdIikKcGx0LnlsaW0oMCwgNTApCnBsdC5ncmlkKCkKCnBsdC5zdWJwbG90KDIsIDQsIDcpCnBsdC5wbG90KHJhbmdlKDg3NjApLCBzb3J0ZWRfSF9kZl9vdXRwdXQyWydFX0VfSF9kX3QgW2tXaC9oXSddLCBjb2xvciA9ICdvcmFuZ2UnKQpwbHQueWxhYmVsKCJBQytGQU7mtojosrvpm7vliptba1doL2hdIikKcGx0LnhsYWJlbCgi5pmC6ZaTW2hdIikKcGx0LnlsaW0oMCwgNSkKcGx0LmdyaWQoKQoKcGx0LnN1YnBsb3QoMiwgNCwgOCkKcGx0LnBsb3QocmFuZ2UoODc2MCksIHNvcnRlZF9IX2RmX291dHB1dDJbJ0VfRV9mYW5fSF9kX3QgW2tXaC9oXSddLCBjb2xvciA9ICdvcmFuZ2UnKQpwbHQueWxhYmVsKCJGQU7mtojosrvpm7vliptba1doL2hdIikKcGx0LnhsYWJlbCgi5pmC6ZaTW2hdIikKcGx0LnlsaW0oMCwgNSkKcGx0LmdyaWQoKQpwbHQuc2hvdygpCgojIyMjQ29vbGluZyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMKCnBsdC5maWd1cmUoZmlnc2l6ZT0oMjUsIDcpKQoKcGx0LnN1YnBsb3QoMiwgMiwgMSkKcGx0LnBsb3Qoc3VtbWVyX2RmX291dHB1dDIuaW5kZXgsCiAgICAgICAgIHN1bW1lcl9kZl9vdXRwdXQyWyJUaGV0YV9leF9kX3QgW+KEg10iXSwKICAgICAgICAgbGFiZWwgPSAi5aSW5rCX5ripW+KEg10iLCBjb2xvciA9ICdsaWdodGJsdWUnKQpwbHQucGxvdChzdW1tZXJfZGZfb3V0cHV0NV9DLmluZGV4LAogICAgICAgICBzdW1tZXJfZGZfb3V0cHV0NV9DWyJYX2V4X2RfdCJdICogMTAwMCwKICAgICAgICAgbGFiZWwgPSAi5aSW5rCX57W25a++5rm/5bqmW2cva2cnXSIsIGNvbG9yID0gJ29yYW5nZScpCnBsdC55bGFiZWwoIua4qeW6plvihINd44CB57W25a++5rm/5bqmW2cva2cnXSIpCnBsdC55bGltKC0xMCwgNDApCnBsdC5ncmlkKCkKcGx0LmxlZ2VuZCgpCgpwbHQuc3VicGxvdCgyLCAyLCAyKQpwbHQucGxvdChzdW1tZXJfZGZfb3V0cHV0Mi5pbmRleCwKICAgICAgICAgKHN1bW1lcl9kZl9vdXRwdXQyWyJxX2hzX0NTX2RfdCBbV2gvaF0iXSArIHN1bW1lcl9kZl9vdXRwdXQyWyJxX2hzX0NMX2RfdCBbV2gvaF0iXSkgLyAxMDAwICwKICAgICAgICAgbGFiZWwgPSAi5Yem55CG54ax6YePW2tXaC9oXSIsIGNvbG9yID0gJ3JlZCcpCnBsdC5wbG90KHN1bW1lcl9kZl9vdXRwdXQyLmluZGV4LAogICAgICAgICBzdW1tZXJfZGZfb3V0cHV0MlsiRV9FX0NfZF90IFtrV2gvaF0iXSwKICAgICAgICAgbGFiZWwgPSAiQUMrRkFO5raI6LK76Zu75YqbW2tXaC9oXSIsIGNvbG9yID0gJ2JsdWUnKQpwbHQucGxvdChzdW1tZXJfZGZfb3V0cHV0Mi5pbmRleCwKICAgICAgICAgc3VtbWVyX2RmX291dHB1dDJbIkVfRV9mYW5fQ19kX3QgW2tXaC9oXSJdLAogICAgICAgICBsYWJlbCA9ICJGQU7mtojosrvpm7vliptba1doL2hdIiwgY29sb3IgPSAnZ3JlZW4nKQpwbHQueWxhYmVsKCLlh6bnkIbnhrHph49ba1doL2hd44CB5raI6LK76Zu75YqbW2tXaC9oXSIpCnBsdC55bGltKDAsIDEwKQpwbHQuZ3JpZCgpCnBsdC5sZWdlbmQoKQoKcGx0LnN1YnBsb3QoMiwgMiwgMykKcGx0LnBsb3Qoc3VtbWVyX2RmX291dHB1dDIuaW5kZXgsCiAgICAgICAgIHN1bW1lcl9kZl9vdXRwdXQyWyJFX1VUX0NfZF90IFtNSi9oXSJdLAogICAgICAgICBsYWJlbCA9ICLlh6bnkIbnhrHph49ba1doL2hdIiwgY29sb3IgPSAncmVkJykKcGx0LnlsYWJlbCgi5pyq5Yem55CG6LKg6I2377yI5LiA5qyh44Ko44ON44Or44Ku44O855u45b2T5YiG77yJW01KL2hdIikKcGx0LnlsaW0oMCwgMTApCnBsdC5ncmlkKCkKcGx0LmxlZ2VuZCgpCgpwbHQuc3VicGxvdCgyLCAyLCA0KQpwbHQucGxvdChzdW1tZXJfZGZfb3V0cHV0Mi5pbmRleCwKICAgICAgICAgKHN1bW1lcl9kZl9vdXRwdXQyWyJxX2hzX0NTX2RfdCBbV2gvaF0iXSArIHN1bW1lcl9kZl9vdXRwdXQyWyJxX2hzX0NMX2RfdCBbV2gvaF0iXSkgLyAxMDAwIC8gc3VtbWVyX2RmX291dHB1dDJbIkVfRV9DX2RfdCBba1doL2hdIl0sCiAgICAgICAgIGxhYmVsID0gInNDT1BbLV0iLCBjb2xvciA9ICdibGFjaycpCnBsdC55bGFiZWwoInNDT1BbLV0iKQpwbHQueWxpbSgwLCAxMCkKcGx0LmdyaWQoKQpwbHQubGVnZW5kKCkKcGx0LnNob3coKQoKIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjCgpwbHQuZmlndXJlKGZpZ3NpemU9KDI1LCA3KSkKCnBsdC5zdWJwbG90KDIsIDQsIDEpCnBsdC5wbG90KHJhbmdlKDg3NjApLCBzb3J0ZWRfQ19kZl9vdXRwdXQyWydxX2hzX0NfZF90IFtXaC9oXSddIC8gMTAwMCwgY29sb3IgPSAnbGlnaHRibHVlJykKcGx0LnlsYWJlbCgi5Yem55CG54ax6YePW2tXaC9oXSIpCnBsdC54bGFiZWwoIuaZgumWk1toXSIpCnBsdC55bGltKDAsIDEwKQpwbHQuZ3JpZCgpCgpwbHQuc3VicGxvdCgyLCA0LCAyKQpwbHQucGxvdChyYW5nZSg4NzYwKSwgc29ydGVkX0NfZGZfb3V0cHV0MlsnVGhldGFfaHNfQ19vdXRfZF90IFvihINdJ10sIGNvbG9yID0gJ2xpZ2h0Ymx1ZScpCnBsdC55bGFiZWwoIuWupOWGheapn+WHuuWPo+a4qeW6plvihINdIikKcGx0LnhsYWJlbCgi5pmC6ZaTW2hdIikKcGx0LnlsaW0oMCwgNTApCnBsdC5ncmlkKCkKCnBsdC5zdWJwbG90KDIsIDQsIDMpCnBsdC5wbG90KHJhbmdlKDg3NjApLCBzb3J0ZWRfQ19kZl9vdXRwdXQyWydUaGV0YV9oc19DX2luX2RfdCBb4oSDXSddLCBjb2xvciA9ICdsaWdodGJsdWUnKQpwbHQueWxhYmVsKCLlrqTlhoXmqZ/lhaXlj6PmuKnluqZb4oSDXSIpCnBsdC54bGFiZWwoIuaZgumWk1toXSIpCnBsdC55bGltKDAsIDUwKQpwbHQuZ3JpZCgpCgpwbHQuc3VicGxvdCgyLCA0LCA0KQpwbHQucGxvdChyYW5nZSg4NzYwKSwgc29ydGVkX0NfZGZfb3V0cHV0MlsnVl9oc19zdXBwbHlfQ19kX3QgW20zL2hdJ10sIGNvbG9yID0gJ2xpZ2h0Ymx1ZScpCnBsdC55bGFiZWwoIuS+m+e1pumiqOmHj1ttMy9oXSIpCnBsdC54bGFiZWwoIuaZgumWk1toXSIpCnBsdC55bGltKDAsIDIwMDApCnBsdC5ncmlkKCkKCnBsdC5zdWJwbG90KDIsIDQsIDUpCnBsdC5wbG90KHJhbmdlKDg3NjApLCBzb3J0ZWRfQ19kZl9vdXRwdXQyWydFX0NfZF90IFtNSi9oXSddLCBjb2xvciA9ICdsaWdodGJsdWUnKQpwbHQueWxhYmVsKCLkuIDmrKHjgqjjg43jg6vjgq7jg7zmtojosrvph49bTUovaF0iKQpwbHQueGxhYmVsKCLmmYLplpNbaF0iKQpwbHQueWxpbSgwLCA1MCkKcGx0LmdyaWQoKQoKcGx0LnN1YnBsb3QoMiwgNCwgNikKcGx0LnBsb3QocmFuZ2UoODc2MCksIHNvcnRlZF9DX2RmX291dHB1dDJbJ0VfVVRfQ19kX3QgW01KL2hdJ10sIGNvbG9yID0gJ2xpZ2h0Ymx1ZScpCnBsdC55bGFiZWwoIuacquWHpueQhuiyoOiNt++8iOS4gOasoeOCqOODjeODq+OCruODvOebuOW9k+WIhu+8iVtNSi9oXSIpCnBsdC54bGFiZWwoIuaZgumWk1toXSIpCnBsdC55bGltKDAsIDUwKQpwbHQuZ3JpZCgpCgpwbHQuc3VicGxvdCgyLCA0LCA3KQpwbHQucGxvdChyYW5nZSg4NzYwKSwgc29ydGVkX0NfZGZfb3V0cHV0MlsnRV9FX0NfZF90IFtrV2gvaF0nXSwgY29sb3IgPSAnbGlnaHRibHVlJykKcGx0LnlsYWJlbCgiQUMrRkFO5raI6LK76Zu75YqbW2tXaC9oXSIpCnBsdC54bGFiZWwoIuaZgumWk1toXSIpCnBsdC55bGltKDAsIDUpCnBsdC5ncmlkKCkKCnBsdC5zdWJwbG90KDIsIDQsIDgpCnBsdC5wbG90KHJhbmdlKDg3NjApLCBzb3J0ZWRfQ19kZl9vdXRwdXQyWydFX0VfZmFuX0NfZF90IFtrV2gvaF0nXSwgY29sb3IgPSAnbGlnaHRibHVlJykKcGx0LnlsYWJlbCgiRkFO5raI6LK76Zu75YqbW2tXaC9oXSIpCnBsdC54bGFiZWwoIuaZgumWk1toXSIpCnBsdC55bGltKDAsIDUpCnBsdC5ncmlkKCkKcGx0LnNob3coKQoKIyMjc0NPUCMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjCnBsdC5maWd1cmUoZmlnc2l6ZT0oMjUsIDcpKQpwbHQuc3VicGxvdCgxLCAyLCAxKQpwbHQuc2NhdHRlcih3aW50ZXJfZGZfb3V0cHV0MlsicV9oc19IX2RfdCBbV2gvaF0iXSAvIDEwMDAsCiAgICAgICAgICAgIHdpbnRlcl9kZl9vdXRwdXQyWyJxX2hzX0hfZF90IFtXaC9oXSJdIC8gMTAwMCAvIHdpbnRlcl9kZl9vdXRwdXQyWyJFX0VfSF9kX3QgW2tXaC9oXSJdLAogICAgICAgICAgICBsYWJlbCA9ICJzQ09QWy1dIiwgY29sb3IgPSAnb3JhbmdlJykKcGx0LnlsYWJlbCgic0NPUFstXSIpCnBsdC55bGltKDAsIDEwKQpwbHQueGxhYmVsKCLlh6bnkIbnhrHph48gW2tXaC9oXSIpCnBsdC54bGltKDAsIDEwKQpwbHQuZ3JpZCgpCnBsdC5sZWdlbmQoKQoKcGx0LnN1YnBsb3QoMSwgMiwgMikKcGx0LnNjYXR0ZXIoKHN1bW1lcl9kZl9vdXRwdXQyWyJxX2hzX0NTX2RfdCBbV2gvaF0iXSArIHN1bW1lcl9kZl9vdXRwdXQyWyJxX2hzX0NMX2RfdCBbV2gvaF0iXSkgLyAxMDAwLAogICAgICAgICAgICAoc3VtbWVyX2RmX291dHB1dDJbInFfaHNfQ1NfZF90IFtXaC9oXSJdICsgc3VtbWVyX2RmX291dHB1dDJbInFfaHNfQ0xfZF90IFtXaC9oXSJdKSAvIDEwMDAgLyBzdW1tZXJfZGZfb3V0cHV0MlsiRV9FX0NfZF90IFtrV2gvaF0iXSwKICAgICAgICAgICAgbGFiZWwgPSAic0NPUFstXSIsIGNvbG9yID0gJ2xpZ2h0Ymx1ZScpCnBsdC55bGFiZWwoInNDT1BbLV0iKQpwbHQueWxpbSgwLCAxMCkKcGx0LnhsYWJlbCgi5Yem55CG54ax6YePIFtrV2gvaF0iKQpwbHQueGxpbSgwLCAxMCkKcGx0LmdyaWQoKQpwbHQubGVnZW5kKCkKCnBsdC5zaG93KCkK").decode("utf-8")

_ASSIGNMENT_RE = re.compile(
    r"^(\s*)([A-Za-z_]\w*)\s*=\s*(.*?)\s+#@para" + r"m\s*(.*)$"
)

def _plain_label(value):
    value = re.sub(r"<[^>]+>", "", value)
    value = value.replace("#@markdown", "").replace("**", "").replace("#", "")
    return html.unescape(value).strip(" -―")

def _safe_literal(expression):
    try:
        return ast.literal_eval(expression)
    except (ValueError, SyntaxError):
        return expression.strip("'\"")

def _extract_options(meta):
    start, end = meta.find("["), meta.rfind("]")
    if start < 0 or end <= start:
        return None
    try:
        return list(ast.literal_eval(meta[start:end + 1]))
    except (ValueError, SyntaxError):
        return None

def _extract_fields(source):
    fields, occurrences = [], defaultdict(int)
    section, group, label = "① 計算条件名", "基本項目", None
    for line in source.splitlines():
        if "#@markdown" in line:
            cleaned = _plain_label(line)
            match = re.search(r"＜([^＞]+)＞", cleaned)
            if match:
                section, group, label = match.group(1).strip(), "基本項目", None
            elif "―" in line and cleaned:
                group, label = cleaned, None
            elif cleaned:
                label = cleaned
        match = _ASSIGNMENT_RE.match(line)
        if not match:
            continue
        _, name, expression, meta = match.groups()
        occurrence = occurrences[name]
        occurrences[name] += 1
        fields.append({
            "id": f"{name}__{occurrence}",
            "name": name,
            "label": label or name,
            "section": section,
            "group": group,
            "default": _safe_literal(expression),
            "options": _extract_options(meta),
        })
    return fields

def _category(section):
    if section.startswith("⑦"):
        return "暖房"
    if section.startswith("⑧"):
        return "冷房"
    if section.startswith("⑨"):
        return "換気"
    return "基本設定"

def _selected_model_index(selected):
    selected = str(selected or "")
    if "ダクト式" in selected:
        return 0
    if "現行省エネ" in selected:
        return 1
    if "潜熱評価モデル" in selected:
        return 2
    if "電中研" in selected or "電力中央研究所" in selected:
        return 3
    return 0

def _model_tab_updates(selected):
    active_index = _selected_model_index(selected)
    return [gr.Tab(interactive=index == active_index) for index in range(4)]

def _section_label(name):
    number_match = re.match(r"^([①-⑨](?:-\d+)?)", name)
    number = number_match.group(1) if number_match else name
    mapping = [
        ("暖房全般", "全般"), ("冷房全般", "全般"),
        ("ダクト式セントラル空調機", "ダクト式"),
        ("現行省エネ法", "RAC・現行"),
        ("潜熱評価モデル", "RAC・潜熱"),
        ("電力中央研究所", "電中研"),
        ("計算条件名", "条件名"), ("外部ファイル名", "外部ファイル"),
        ("計算時定数", "計算定数"), ("基本情報", "基本情報"),
        ("外皮条件", "外皮条件"), ("過剰熱量持越し", "熱量持越し"),
        ("その他", "その他"), ("熱交換型換気", "熱交換換気"),
    ]
    for keyword, description in mapping:
        if keyword in name:
            return f"{number} {description}"
    return number

_FIELDS = _extract_fields(_FORM_SOURCE)
if len(_FIELDS) != 222:
    raise RuntimeError(f"入力定義数が想定と異なります: {len(_FIELDS)}")

def _coerce(field, value):
    default = field["default"]
    if value is None:
        return default
    if isinstance(default, bool):
        return bool(value)
    if isinstance(default, int) and not isinstance(default, bool):
        return int(value)
    if isinstance(default, float):
        return float(value)
    return value

def _build_input_data(args):
    values = {
        field["id"]: _coerce(field, value)
        for field, value in zip(_FIELDS, args)
    }
    transformed, occurrences = [], defaultdict(int)
    for line in _FORM_SOURCE.splitlines():
        if "jjjexperiment.main.calc(input_data)" in line:
            continue
        match = _ASSIGNMENT_RE.match(line)
        if match:
            indent, name, expression, _ = match.groups()
            occurrence = occurrences[name]
            occurrences[name] += 1
            field_id = f"{name}__{occurrence}"
            line = f"{indent}{name} = _values.get({field_id!r}, {expression})"
        transformed.append(line)
    namespace = {"_values": values}
    exec("\n".join(transformed), namespace)
    return namespace["input_data"]

def _component(field):
    label = f"{field['label']} [{field['name']}]"
    default, options = field["default"], field["options"]
    if options:
        return gr.Dropdown(choices=options, value=default, label=label)
    if isinstance(default, bool):
        return gr.Checkbox(value=default, label=label)
    if isinstance(default, int):
        return gr.Number(value=default, precision=0, label=label)
    if isinstance(default, float):
        return gr.Number(value=default, label=label)
    return gr.Textbox(value=str(default), label=label)

def _check_ui(*args):
    try:
        data = _build_input_data(args)
        return "✅ 入力内容を作成しました（計算は未実行です）。", data
    except Exception:
        return "❌ 入力エラー\n\n" + traceback.format_exc(), None

class _TeeWriter:
    def __init__(self, *streams):
        self.streams = streams
        self.encoding = getattr(streams[0], "encoding", "utf-8") if streams else "utf-8"

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()

def _display_log(text, max_chars=100000):
    text = text or ""
    if not text.strip():
        return "（標準出力・標準エラーはありませんでした）"
    if len(text) <= max_chars:
        return text
    return (
        f"※ ログが長いため末尾{max_chars:,}文字を表示しています。"
        "全文は計算出力ファイルの *_gradio.log を確認してください。\n\n"
        + text[-max_chars:]
    )

def _save_log(data, text):
    if not data:
        return None
    try:
        prefix = f"{data.get('case_name', 'default')}{version_info()}"
        path = Path.cwd() / f"{prefix}_gradio.log"
        path.write_text(text or "", encoding="utf-8")
        return str(path.resolve())
    except Exception as error:
        print(f"[Gradio] 計算ログを保存できませんでした: {error}", file=sys.stderr)
        return None

def _calculation_files(data):
    if not data:
        return []
    prefix = f"{data.get('case_name', 'default')}{version_info()}"
    return [
        str(path.resolve())
        for path in sorted(Path.cwd().glob(prefix + "*"))
        if path.is_file() and path.suffix.lower() != ".zip"
    ]

def _run_ui(*args, progress=gr.Progress()):
    started_at = time.perf_counter()
    data = None
    log_buffer = io.StringIO()
    yield "**状態：⏳ 計算準備中** — 入力内容を確認しています。", None, None, [], "計算開始待ち"
    try:
        progress(0.05, desc="入力内容を確認しています")
        data = _build_input_data(args)
        yield (
            "**状態：⏳ 計算中** — 完了するまでこの画面を閉じずにお待ちください。",
            data,
            None,
            [],
            "計算処理を実行しています。出力は完了後に表示します。",
        )
        progress(0.15, desc="計算を実行しています")
        with contextlib.redirect_stdout(_TeeWriter(sys.stdout, log_buffer)), \
             contextlib.redirect_stderr(_TeeWriter(sys.stderr, log_buffer)):
            jjjexperiment.main.calc(data)
        log_text = log_buffer.getvalue()
        _save_log(data, log_text)
        result_paths = _calculation_files(data)
        progress(1.0, desc="計算が完了しました")
        elapsed = time.perf_counter() - started_at
        yield (
            f"**状態：✅ 計算完了** — 経過時間：{elapsed:.1f}秒",
            data,
            data,
            result_paths,
            _display_log(log_text),
        )
    except Exception:
        error_text = traceback.format_exc()
        log_text = log_buffer.getvalue() + "\n" + error_text
        _save_log(data, log_text)
        result_paths = _calculation_files(data) if data else []
        elapsed = time.perf_counter() - started_at
        yield (
            f"**状態：❌ 計算エラー** — 経過時間：{elapsed:.1f}秒\n\n"
            + error_text,
            None,
            None,
            result_paths,
            _display_log(log_text),
        )

def _local_paths(files):
    if files is None:
        return []
    if isinstance(files, (str, os.PathLike)):
        return [str(files)]
    result = []
    for item in files:
        if isinstance(item, (str, os.PathLike)):
            result.append(str(item))
        elif hasattr(item, "name"):
            result.append(item.name)
        elif isinstance(item, dict):
            result.append(item.get("path") or item.get("name"))
    return [path for path in result if path]

def _run_local(files, progress=gr.Progress()):
    started_at = time.perf_counter()
    paths = _local_paths(files)
    if not paths:
        yield (
            "**状態：❌ 未実行** — JSONファイルをアップロードしてください。",
            None,
            None,
            [],
            "計算は実行されていません。",
        )
        return
    total = len(paths)
    last = None
    result_paths = []
    combined_log = io.StringIO()
    yield (
        f"**状態：⏳ 計算準備中** — {total}件のJSONを確認しています。",
        None,
        None,
        [],
        "計算開始待ち",
    )
    try:
        for index, path in enumerate(paths, start=1):
            name = os.path.basename(path)
            progress((index - 1, total), desc=f"{index}/{total}件目を計算しています")
            with open(path, encoding="utf-8-sig") as file:
                last = json.load(file)
            yield (
                f"**状態：⏳ 計算中** — {index}/{total}件目：{name}",
                last,
                None,
                result_paths,
                _display_log(combined_log.getvalue()) if combined_log.tell() else
                f"{index}/{total}件目を計算しています。出力は完了後に表示します。",
            )
            item_log = io.StringIO()
            with contextlib.redirect_stdout(_TeeWriter(sys.stdout, item_log)), \
                 contextlib.redirect_stderr(_TeeWriter(sys.stderr, item_log)):
                jjjexperiment.main.calc(last)
            item_text = item_log.getvalue()
            combined_log.write(f"===== {index}/{total}: {name} =====\n")
            combined_log.write(item_text)
            if item_text and not item_text.endswith("\n"):
                combined_log.write("\n")
            combined_log.write("\n")
            _save_log(last, item_text)
            for result_path in _calculation_files(last):
                if result_path not in result_paths:
                    result_paths.append(result_path)
        progress(1.0, desc="すべての計算が完了しました")
        elapsed = time.perf_counter() - started_at
        yield (
            f"**状態：✅ 計算完了** — {total}件、経過時間：{elapsed:.1f}秒",
            last,
            last,
            result_paths,
            _display_log(combined_log.getvalue()),
        )
    except Exception:
        error_text = traceback.format_exc()
        if 'item_log' in locals():
            combined_log.write(item_log.getvalue())
        combined_log.write("\n===== 計算エラー =====\n")
        combined_log.write(error_text)
        _save_log(last, combined_log.getvalue())
        if last:
            for result_path in _calculation_files(last):
                if result_path not in result_paths:
                    result_paths.append(result_path)
        elapsed = time.perf_counter() - started_at
        yield (
            f"**状態：❌ 計算エラー** — 経過時間：{elapsed:.1f}秒\n\n"
            + error_text,
            None,
            None,
            result_paths,
            _display_log(combined_log.getvalue()),
        )

def _graphs(data):
    if not data:
        return ["❌ 先に計算してください。"] + [None] * 5
    figures, original_show = [], plt.show
    def capture(*args, **kwargs):
        figures.append(plt.gcf())
    try:
        plt.close("all")
        plt.show = capture
        exec(compile(_GRAPH_SOURCE, "graphs.py", "exec"), {"input_data": data})
    except Exception:
        return ["❌ グラフエラー\n\n" + traceback.format_exc()] + [None] * 5
    finally:
        plt.show = original_show
    figures = (figures + [None] * 5)[:5]
    return ["✅ 最後の計算結果を表示します。"] + figures

def _graph_panel(state):
    gr.Markdown("> 計算完了後に自動更新します。複数計算時は最後の計算結果を表示します。")
    button = gr.Button("グラフを再表示")
    status = gr.Markdown("計算完了後、ここにグラフの状態を表示します。")
    plots = [gr.Plot(label=label) for label in [
        "暖房時系列", "暖房負荷順", "冷房時系列", "冷房負荷順", "sCOP"
    ]]
    outputs = [status, *plots]
    button.click(_graphs, inputs=state, outputs=outputs, show_progress="full")
    return outputs

def _downloads(result_paths):
    paths = [Path(path) for path in (result_paths or []) if Path(path).is_file()]
    if not paths:
        return "出力ファイルはまだありません。", None, []
    download_dir = Path(tempfile.mkdtemp(prefix="verification_platform_"))
    archive = download_dir / "verification_platform_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for path in paths:
            zip_file.write(path, arcname=path.name)
    return (
        f"✅ {len(paths)}個の出力ファイルをダウンロードできます。",
        str(archive),
        [str(path) for path in paths],
    )

def _log_panel():
    gr.Markdown("> Colabへ出力される標準出力・標準エラーを、ここでも確認できます。")
    return gr.Textbox(
        value="未実行",
        label="標準出力・標準エラー",
        lines=18,
        max_lines=40,
        interactive=False,
    )

def _download_panel(result_files):
    gr.Markdown("> 一括ダウンロードはZIP、必要なファイルだけ取得する場合は個別ファイルを使用してください。")
    button = gr.Button("ダウンロード一覧を再作成")
    status = gr.Markdown("計算完了後、ここにダウンロード欄を表示します。")
    archive = gr.File(label="一括ダウンロード（ZIP）", interactive=False)
    files = gr.File(
        label="個別ファイル",
        file_count="multiple",
        interactive=False,
    )
    outputs = [status, archive, files]
    button.click(_downloads, inputs=result_files, outputs=outputs, show_progress="full")
    return outputs

def _result_area(state, result_files, preview_label):
    gr.Markdown("### 計算内容・結果")
    gr.Markdown("> 計算を実行すると、入力内容・ログ・グラフ・出力ファイルを自動更新します。")
    with gr.Tabs():
        with gr.Tab("入力内容"):
            gr.Markdown("> 計算に使用する input_data です。計算実行時にも自動表示します。")
            preview = gr.JSON(label=preview_label)
        with gr.Tab("計算ログ", render_children=True):
            log_output = _log_panel()
        with gr.Tab("グラフ", render_children=True):
            graph_outputs = _graph_panel(state)
        with gr.Tab("出力ファイル", render_children=True):
            download_outputs = _download_panel(result_files)
    return preview, log_output, graph_outputs, download_outputs

with gr.Blocks(title="Verification Platform 260715 Gradio") as demo:
    gr.Markdown("# Verification Platform 260715版 — Gradio UI")
    gr.Markdown("入力方法を選択してください。UIで直接入力するか、端末内のJSONファイルをアップロードして計算できます。")
    state = gr.State(value=None)
    result_files = gr.State(value=[])
    with gr.Tabs():
        with gr.Tab("(1) UI入力"):
            gr.Markdown("### 第1階層")
            components = OrderedDict()
            heating_model_tabs = [None] * 4
            cooling_model_tabs = [None] * 4
            with gr.Tabs():
                for category in ["基本設定", "暖房", "冷房", "換気"]:
                    category_fields = [f for f in _FIELDS if _category(f["section"]) == category]
                    with gr.Tab(f"{category} ({len(category_fields)}項目)"):
                        gr.Markdown("### 第2階層")
                        if category in ("暖房", "冷房"):
                            gr.Markdown(
                                "> 「全般」で選択した設備方式に対応する方式タブだけが有効になります。"
                            )
                        sections = OrderedDict()
                        for field in category_fields:
                            sections.setdefault(field["section"], []).append(field)
                        with gr.Tabs():
                            for section_name, section_fields in sections.items():
                                model_match = re.match(r"^[⑦⑧]-([1-4])", section_name)
                                model_index = int(model_match.group(1)) - 1 if model_match else None
                                initially_interactive = model_index in (None, 0)
                                with gr.Tab(
                                    f"{_section_label(section_name)} ({len(section_fields)}項目)",
                                    interactive=initially_interactive,
                                ) as section_tab:
                                    gr.Markdown(f"### {section_name}")
                                    current_group = None
                                    for field in section_fields:
                                        if field["group"] != current_group:
                                            current_group = field["group"]
                                            if current_group != "基本項目":
                                                gr.Markdown(f"#### {current_group}")
                                        components[field["id"]] = _component(field)
                                if model_index is not None:
                                    if section_name.startswith("⑦-"):
                                        heating_model_tabs[model_index] = section_tab
                                    elif section_name.startswith("⑧-"):
                                        cooling_model_tabs[model_index] = section_tab
            if any(tab is None for tab in heating_model_tabs + cooling_model_tabs):
                raise RuntimeError("暖房・冷房の方式タブをすべて作成できませんでした。")
            components["H_A_type__0"].change(
                _model_tab_updates,
                inputs=components["H_A_type__0"],
                outputs=heating_model_tabs,
                queue=False,
                show_progress="hidden",
            )
            components["C_A_type__0"].change(
                _model_tab_updates,
                inputs=components["C_A_type__0"],
                outputs=cooling_model_tabs,
                queue=False,
                show_progress="hidden",
            )
            ordered_components = [components[field["id"]] for field in _FIELDS]
            status = gr.Markdown("**状態：未実行**")
            with gr.Row():
                check = gr.Button("計算せず入力内容を確認")
                run = gr.Button("▶ 計算を実行", variant="primary")
            preview, log_output, graph_outputs, download_outputs = _result_area(
                state,
                result_files,
                "入力内容（input_data）",
            )
            check.click(_check_ui, ordered_components, [status, preview])
            run_event = run.click(
                _run_ui,
                ordered_components,
                [status, preview, state, result_files, log_output],
                show_progress="full",
                show_progress_on=status,
                scroll_to_output=True,
                concurrency_limit=1,
                concurrency_id="calculation",
            )
            run_event.then(
                _graphs,
                state,
                graph_outputs,
                show_progress="full",
                show_progress_on=graph_outputs[0],
            )
            run_event.then(
                _downloads,
                result_files,
                download_outputs,
                show_progress="full",
                show_progress_on=download_outputs[0],
            )

        with gr.Tab("(2) ローカル"):
            gr.Markdown(
                "端末内のJSONファイルをアップロードして計算します。"
                "複数ファイルを指定した場合、入力内容とグラフには最後に計算した結果だけを表示します。"
            )
            files = gr.File(
                file_count="multiple", file_types=[".json"],
                type="filepath", label="JSONファイル（複数選択可）"
            )
            status2 = gr.Markdown("**状態：未実行**")
            run2 = gr.Button("▶ アップロードしたJSONを計算", variant="primary")
            preview2, log_output2, graph_outputs2, download_outputs2 = _result_area(
                state,
                result_files,
                "最後に計算した入力内容（input_data）",
            )
            run_event2 = run2.click(
                _run_local,
                files,
                [status2, preview2, state, result_files, log_output2],
                show_progress="full",
                show_progress_on=status2,
                scroll_to_output=True,
                concurrency_limit=1,
                concurrency_id="calculation",
            )
            run_event2.then(
                _graphs,
                state,
                graph_outputs2,
                show_progress="full",
                show_progress_on=graph_outputs2[0],
            )
            run_event2.then(
                _downloads,
                result_files,
                download_outputs2,
                show_progress="full",
                show_progress_on=download_outputs2[0],
            )

demo.queue().launch(share=True, debug=True)
